In [2]:
# Notebook 全局导入与超参数。只改这一格，然后从上到下重新运行整个 notebook。
from pathlib import Path
import importlib.util
import sys
import warnings

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
if support_spec is None or support_spec.loader is None:
    raise ImportError(f"无法加载训练辅助模块: {SUPPORT_PATH}")

cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

PROJECT_ROOT = SUPPORT_PATH.parent

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb

# NOTEBOOK_RANDOM_SEED: 控制数据切分、BayesSearch、SMOTE、early stopping 和模型训练的随机种子。
NOTEBOOK_RANDOM_SEED = 114514
# NOTEBOOK_TEST_SIZE: 固定留作最终评估的测试集比例。
NOTEBOOK_TEST_SIZE = 0.15
# NOTEBOOK_BAYES_N_ITER: BayesSearchCV 的候选参数组合采样次数。
NOTEBOOK_BAYES_N_ITER = 24
# NOTEBOOK_CV_FOLDS: 分层交叉验证折数。
NOTEBOOK_CV_FOLDS = 5
# NOTEBOOK_MODEL_N_JOBS: 单个 LightGBM 模型内部使用的线程数。
NOTEBOOK_MODEL_N_JOBS = 1
# NOTEBOOK_SEARCH_N_JOBS: BayesSearchCV 并行 worker 数；GPU 搜索通常保持 1 更稳。
NOTEBOOK_SEARCH_N_JOBS = 1
# NOTEBOOK_SMOTE_K_NEIGHBORS: 当策略为 smote 时，每个训练折内部 SMOTE 的近邻数。
NOTEBOOK_SMOTE_K_NEIGHBORS = 3
# NOTEBOOK_SCALE_POS_WEIGHT: 当策略为 scale_pos_weight 时传给 LightGBM 的正类权重；None 表示按折内训练子集自动计算。
NOTEBOOK_SCALE_POS_WEIGHT = None
# NOTEBOOK_EARLY_STOPPING_ROUNDS: 每个训练折内部用于 early stopping 的 patience。
NOTEBOOK_EARLY_STOPPING_ROUNDS = 100
# NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION: 每个训练折内部再切出的 early stopping 验证集比例。
NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION = 0.15
# NOTEBOOK_BALANCE_STRATEGIES: 本轮要做 A/B 对比的类别平衡策略。
NOTEBOOK_BALANCE_STRATEGIES = ("smote", "scale_pos_weight")
# NOTEBOOK_BAYES_SCORING: BayesSearchCV 选参使用的目标指标。
NOTEBOOK_BAYES_SCORING = "roc_auc"
# NOTEBOOK_BAYES_VERBOSE: BayesSearchCV 日志级别。
NOTEBOOK_BAYES_VERBOSE = 0
# NOTEBOOK_TQDM_DESC: notebook 中训练进度条的标题。
NOTEBOOK_TQDM_DESC = "LightGBM BayesSearchCV"
# NOTEBOOK_THRESHOLD_SELECTION_METRIC: 用训练集 OOF 概率挑选最终分类阈值时的主指标。
NOTEBOOK_THRESHOLD_SELECTION_METRIC = "F1"
# NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS: 阈值探测时扫描的阈值列表。
NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS = np.round(np.linspace(0.30, 0.70, 41), 3).tolist()
NOTEBOOK_MODEL_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_model.txt"
NOTEBOOK_PREPROCESSOR_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_preprocessor.joblib"
NOTEBOOK_MANIFEST_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_inference_assets.json"

NOTEBOOK_LGBM_SEARCH_SPACES = {
    "num_leaves": Integer(24, 63),  # 单棵树的最大叶子数；当前最优解曾贴边，扩大上界后继续搜索。
    "learning_rate": Real(3e-3, 1.0e-2, prior="log-uniform"),  # learning rate 保持偏小，配合更多树和 early stopping。
    "n_estimators": Integer(1200, 3200),  # 允许更长 boosting，但交给 early stopping 截断。
    "max_depth": Categorical([5, 6, 7]),  # 允许比上一轮略深，但不放开到无限深。
    "subsample": Real(0.65, 0.90),  # 行采样略放宽，同时保留随机性以抑制过拟合。
    "colsample_bytree": Real(0.65, 0.90),  # 列采样略放宽，降低特征共适应。
    "min_child_samples": Integer(80, 220),  # 抬高下界，让叶子分裂保持更保守。
    "min_split_gain": Real(0.02, 0.20, prior="log-uniform"),  # 抬高下界，减少边际收益很低的分裂。
    "reg_alpha": Real(0.3, 6.0, prior="log-uniform"),  # 适度提高 L1 正则的下界。
    "reg_lambda": Real(8.0, 40.0, prior="log-uniform"),  # 适度提高 L2 正则的下界。
}

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config(
    bayes_n_iter=NOTEBOOK_BAYES_N_ITER,
    cv_folds=NOTEBOOK_CV_FOLDS,
    random_seed=NOTEBOOK_RANDOM_SEED,
    test_size=NOTEBOOK_TEST_SIZE,
    model_n_jobs=NOTEBOOK_MODEL_N_JOBS,
    search_n_jobs=NOTEBOOK_SEARCH_N_JOBS,
    smote_k_neighbors=NOTEBOOK_SMOTE_K_NEIGHBORS,
    early_stopping_rounds=NOTEBOOK_EARLY_STOPPING_ROUNDS,
    early_stopping_validation_fraction=NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION,
)
NOTEBOOK_CV = StratifiedKFold(
    n_splits=NOTEBOOK_CONFIG.cv_folds,
    shuffle=True,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)
print(f"平衡策略 A/B: {NOTEBOOK_BALANCE_STRATEGIES}")
print(f"阈值选择指标: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")
print(f"阈值候选数: {len(NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS)}")
print(f"模型输出路径: {NOTEBOOK_MODEL_OUTPUT_PATH}")
print(f"预处理输出路径: {NOTEBOOK_PREPROCESSOR_OUTPUT_PATH}")
print(f"推理清单路径: {NOTEBOOK_MANIFEST_OUTPUT_PATH}")


随机种子: 114514
数据文件: /root/lightgbm/质谱数据汇总_处理后后后2.csv
测试集比例: 0.15
BayesSearch n_iter: 24
CV folds: 5
模型 n_jobs: 1
搜索 n_jobs: 1
SMOTE k_neighbors: 3
折内 early stopping rounds: 100
折内 early stopping 验证集比例: 0.15
平衡策略 A/B: ('smote', 'scale_pos_weight')
阈值选择指标: F1
阈值候选数: 41
模型输出路径: /root/models/lightgbm_cuda_model.txt
预处理输出路径: /root/models/lightgbm_cuda_preprocessor.joblib
推理清单路径: /root/models/lightgbm_cuda_inference_assets.json


## LightGBM（CUDA-only）

- 第一个代码单元负责 import、路径修正、notebook 全局配置、A/B 策略配置、阈值探测配置，以及模型输出路径。
- 第二个代码单元只做数据加载、`RETENTION_TIME` 清洗诊断、训练/测试集切分，不启动训练。
- 第三个代码单元会分别对 `smote` 与 `scale_pos_weight` 跑一轮 `BayesSearchCV`，统一启用折内 early stopping，并按 CV AUC 选出最终策略。
- 第四个代码单元会基于训练集 OOF 概率做阈值探测，避免直接拿测试集选阈值；选好阈值后再保存完整推理资产。
- LightGBM 训练设备固定为 `cuda`，禁止 CPU fallback。
- 缺失值填充、标准化和类别平衡都在 estimator 的 `fit` 内部执行；BayesSearchCV 的每个训练折都会单独拟合预处理，并在折内训练子集上做 SMOTE 或 `scale_pos_weight`。
- `RETENTION_TIME` 会统一解析：普通数值直接保留，多值文本（如 `17.9 and 18.5`）按均值折叠为单值，真正缺失值保留为缺失，后续再由训练折内众数填补。
- Notebook 训练进度条由 `tqdm.auto` 提供。
- 训练完成后会同时保存 `models/lightgbm_cuda_model.txt`、`models/lightgbm_cuda_preprocessor.joblib` 和 `models/lightgbm_cuda_inference_assets.json`。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip install -r requirements.txt
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```


In [3]:
# 加载数据、切分数据并输出训练前诊断。
random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
)
retention_time_raw = data["RETENTION_TIME"].copy()

prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
)
X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
retention_time_diagnostics = prepared["retention_time_diagnostics"]

print("=== 训练前诊断 ===")
print("样本总数:", len(data))
print("标签分布:")
print(data[DEFAULT_TARGET_COLUMN].value_counts())
print()
print("转换前 RETENTION_TIME 的示例值：")
print(retention_time_raw.head())
print()
if retention_time_diagnostics is not None:
    print("RETENTION_TIME 清洗诊断：")
    print(f"严格数值转换后的缺失数: {retention_time_diagnostics['strict_missing_count']}")
    print(f"清洗后的缺失数        : {retention_time_diagnostics['cleaned_missing_count']}")
    print(f"从文本中恢复的记录数  : {retention_time_diagnostics['recovered_from_text_count']}")
    print(f"多值文本记录数        : {retention_time_diagnostics['multi_value_count']}")
    print(f"原始真实缺失数        : {retention_time_diagnostics['original_missing_count']}")
    print(f"仍无法解析的非空记录数: {retention_time_diagnostics['unparsed_non_missing_count']}")
    print("多值文本示例：")
    print(retention_time_diagnostics["multi_value_examples"] or ["<none>"])
    print()
print("训练集形状（原始；fold 内平衡策略在 fit 时执行）:", X_train.shape)
print("测试集形状:", X_test.shape)
print("训练集标签分布（原始）:")
print(pd.Series(y_train).value_counts())
print()
print("测试集标签分布:")
print(pd.Series(y_test).value_counts())
print()

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print("LightGBM CUDA preflight:", lgbm_version)


=== 训练前诊断 ===
样本总数: 16806
标签分布:
0    10920
1     5886
Name: 毒性, dtype: int64

转换前 RETENTION_TIME 的示例值：
0    4.07435
1      2.354
2        NaN
3      7.557
4     13.870
Name: RETENTION_TIME, dtype: object

RETENTION_TIME 清洗诊断：
严格数值转换后的缺失数: 3266
清洗后的缺失数        : 3260
从文本中恢复的记录数  : 6
多值文本记录数        : 6
原始真实缺失数        : 3260
仍无法解析的非空记录数: 0
多值文本示例：
['17.9  and 18.5']

训练集形状（原始；fold 内平衡策略在 fit 时执行）: (14285, 14)
测试集形状: (2521, 14)
训练集标签分布（原始）:
0    9282
1    5003
Name: 毒性, dtype: int64

测试集标签分布:
0    1638
1     883
Name: 毒性, dtype: int64

LightGBM CUDA preflight: 4.6.0


In [ ]:
# A/B 训练块：比较 SMOTE 与 scale_pos_weight，按 CV AUC 选出最终策略。
metric_order = [
    "AUC",
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
]


def print_metric_block(title, metrics):
    print(title)
    for metric in metric_order:
        print(f"{metric:<18}: {metrics[metric]:.4f}")
    print()


strategy_results = {}
strategy_summary_rows = []

for balance_strategy in NOTEBOOK_BALANCE_STRATEGIES:
    print(f"=== 开始策略: {balance_strategy} ===")
    lgb_model = cuda_training_support.build_lgbm_classifier(
        random_state=NOTEBOOK_CONFIG.random_seed,
        model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
        balance_strategy=balance_strategy,
        smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
        scale_pos_weight=NOTEBOOK_SCALE_POS_WEIGHT,
        early_stopping_rounds=NOTEBOOK_CONFIG.early_stopping_rounds,
        early_stopping_validation_fraction=NOTEBOOK_CONFIG.early_stopping_validation_fraction,
    )

    bayes_search = BayesSearchCV(
        estimator=lgb_model,
        search_spaces=NOTEBOOK_LGBM_SEARCH_SPACES,
        n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
        cv=NOTEBOOK_CV,
        scoring=NOTEBOOK_BAYES_SCORING,
        n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
        verbose=NOTEBOOK_BAYES_VERBOSE,
        random_state=NOTEBOOK_CONFIG.random_seed,
    )

    progress_bar = tqdm(
        total=NOTEBOOK_CONFIG.bayes_n_iter,
        desc=f"{NOTEBOOK_TQDM_DESC} [{balance_strategy}]",
        unit="iter",
    )
    progress_state = {"completed": 0}


    def update_training_progress(_optim_result):
        progress_state["completed"] += 1
        progress_bar.update(1)
        progress_bar.set_postfix(completed=progress_state["completed"], refresh=False)
        return False


    try:
        bayes_search.fit(X_train, y_train, callback=update_training_progress)
    finally:
        progress_bar.close()

    best_estimator = bayes_search.best_estimator_
    y_train_proba = best_estimator.predict_proba(X_train)[:, 1]
    y_test_proba = best_estimator.predict_proba(X_test)[:, 1]
    train_metrics = cuda_training_support.compute_binary_classification_metrics(
        y_true=y_train,
        positive_proba=y_train_proba,
        threshold=0.5,
    )
    test_metrics = cuda_training_support.compute_binary_classification_metrics(
        y_true=y_test,
        positive_proba=y_test_proba,
        threshold=0.5,
    )

    strategy_results[balance_strategy] = {
        "strategy": balance_strategy,
        "bayes_search": bayes_search,
        "best_estimator": best_estimator,
        "best_params": dict(bayes_search.best_params_),
        "cv_auc": float(bayes_search.best_score_),
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "y_train_proba": y_train_proba,
        "y_test_proba": y_test_proba,
        "effective_scale_pos_weight": getattr(best_estimator, "effective_scale_pos_weight_", None),
        "best_iteration": getattr(best_estimator, "best_iteration_", None),
        "fit_class_counts": getattr(best_estimator, "fit_class_counts_", {}),
        "train_split_class_counts": getattr(best_estimator, "train_split_class_counts_", {}),
        "eval_split_class_counts": getattr(best_estimator, "eval_split_class_counts_", {}),
        "model_fit_class_counts": getattr(best_estimator, "model_fit_class_counts_", {}),
    }
    strategy_summary_rows.append(
        {
            "strategy": balance_strategy,
            "cv_auc": float(bayes_search.best_score_),
            "test_auc": test_metrics["AUC"],
            "test_f1": test_metrics["F1"],
            "test_balanced_accuracy": test_metrics["Balanced Accuracy"],
            "best_iteration": getattr(best_estimator, "best_iteration_", None),
            "effective_scale_pos_weight": getattr(best_estimator, "effective_scale_pos_weight_", None),
        }
    )

    print("最佳参数组合:", bayes_search.best_params_)
    print("最佳交叉验证AUC:", bayes_search.best_score_)
    print("最终 refit 前训练集标签分布:", getattr(best_estimator, "fit_class_counts_", {}))
    print("折内训练子集标签分布:", getattr(best_estimator, "train_split_class_counts_", {}))
    print("折内 early stopping 验证集标签分布:", getattr(best_estimator, "eval_split_class_counts_", {}))
    print("最终模型拟合标签分布:", getattr(best_estimator, "model_fit_class_counts_", {}))
    print("effective scale_pos_weight:", getattr(best_estimator, "effective_scale_pos_weight_", None))
    print("best_iteration:", getattr(best_estimator, "best_iteration_", None))
    print_metric_block("=== 训练集性能（threshold=0.5） ===", train_metrics)
    print_metric_block("=== 测试集性能（threshold=0.5） ===", test_metrics)

strategy_comparison = pd.DataFrame(strategy_summary_rows).sort_values(
    by=["cv_auc", "strategy"],
    ascending=[False, True],
).reset_index(drop=True)
print("=== 平衡策略对比（按 CV AUC 排序） ===")
display(strategy_comparison)

best_balance_strategy = strategy_comparison.loc[0, "strategy"]
best_run = strategy_results[best_balance_strategy]
best_lgb = best_run["best_estimator"]
best_y_train_proba = best_run["y_train_proba"]
best_y_test_proba = best_run["y_test_proba"]
default_threshold = 0.5
default_train_metrics = best_run["train_metrics"]
default_test_metrics = best_run["test_metrics"]
default_test_confusion_matrix = default_test_metrics["confusion_matrix"]

print(f"选中的平衡策略: {best_balance_strategy}")
print(f"选择依据: 最高 CV AUC ({best_run['cv_auc']:.4f})")
print(f"best_iteration: {best_run['best_iteration']}")
print_metric_block("=== 选中策略的训练集性能（threshold=0.5） ===", default_train_metrics)
print_metric_block("=== 选中策略的测试集性能（threshold=0.5） ===", default_test_metrics)
print("测试集混淆矩阵（threshold=0.5）:")
print(default_test_confusion_matrix)

plt.figure(figsize=(10, 6))
lgb.plot_importance(best_lgb.booster_, max_num_features=20)
plt.tight_layout()
plt.show()


=== 开始策略: smote ===


LightGBM BayesSearchCV [smote]:   0%|          | 0/24 [00:00<?, ?iter/s]

In [ ]:
# 阈值探测块：基于训练集 OOF 概率选择阈值，再在独立测试集上汇报最终结果并保存推理资产。
allowed_threshold_metrics = {
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
}
if NOTEBOOK_THRESHOLD_SELECTION_METRIC not in allowed_threshold_metrics:
    raise ValueError(
        f"NOTEBOOK_THRESHOLD_SELECTION_METRIC 必须属于 {sorted(allowed_threshold_metrics)}"
    )

oof_train_proba = np.zeros(len(y_train), dtype=float)
oof_progress = tqdm(total=NOTEBOOK_CONFIG.cv_folds, desc="Threshold OOF CV", unit="fold")

try:
    for fold_idx, (train_idx, valid_idx) in enumerate(NOTEBOOK_CV.split(X_train, y_train), start=1):
        fold_model = clone(best_lgb)
        X_fold_train = X_train.iloc[train_idx].reset_index(drop=True)
        y_fold_train = y_train.iloc[train_idx].reset_index(drop=True)
        X_fold_valid = X_train.iloc[valid_idx].reset_index(drop=True)

        fold_model.fit(X_fold_train, y_fold_train)
        oof_train_proba[valid_idx] = fold_model.predict_proba(X_fold_valid)[:, 1]
        oof_progress.update(1)
        oof_progress.set_postfix(fold=fold_idx, refresh=False)
finally:
    oof_progress.close()

threshold_probe = cuda_training_support.probe_binary_classification_thresholds(
    y_true=y_train,
    positive_proba=oof_train_proba,
    thresholds=NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS,
)


def select_threshold_row(threshold_frame, primary_metric):
    priority_map = {
        "F1": ["F1", "Balanced Accuracy", "Recall", "Specificity"],
        "Balanced Accuracy": ["Balanced Accuracy", "F1", "Recall", "Specificity"],
        "Recall": ["Recall", "F1", "Balanced Accuracy", "Specificity"],
        "Precision": ["Precision", "F1", "Balanced Accuracy", "Recall"],
        "Specificity": ["Specificity", "Balanced Accuracy", "F1", "Recall"],
        "Accuracy": ["Accuracy", "Balanced Accuracy", "F1", "Recall"],
    }
    sort_by = priority_map[primary_metric] + ["threshold"]
    ascending = [False] * len(priority_map[primary_metric]) + [True]
    return threshold_frame.sort_values(by=sort_by, ascending=ascending).iloc[0]


top_f1_thresholds = threshold_probe.sort_values(
    by=["F1", "Balanced Accuracy", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
top_balanced_thresholds = threshold_probe.sort_values(
    by=["Balanced Accuracy", "F1", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
selected_threshold_row = select_threshold_row(
    threshold_probe,
    NOTEBOOK_THRESHOLD_SELECTION_METRIC,
)
SELECTED_THRESHOLD = float(selected_threshold_row["threshold"])

print("=== 训练集 OOF 阈值探测：按 F1 排名前 10 ===")
display(top_f1_thresholds)
print("=== 训练集 OOF 阈值探测：按 Balanced Accuracy 排名前 10 ===")
display(top_balanced_thresholds)
print(f"选定阈值: {SELECTED_THRESHOLD:.3f}")
print(f"阈值选择指标: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")

selected_oof_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_train,
    positive_proba=oof_train_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_test,
    positive_proba=best_y_test_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_confusion_matrix = selected_test_metrics["confusion_matrix"]

saved_artifacts = cuda_training_support.save_lightgbm_inference_artifacts(
    estimator=best_lgb,
    prepared=prepared,
    model_path=NOTEBOOK_MODEL_OUTPUT_PATH,
    preprocessor_path=NOTEBOOK_PREPROCESSOR_OUTPUT_PATH,
    manifest_path=NOTEBOOK_MANIFEST_OUTPUT_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    data_path=DATA_PATH,
    random_seed=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=(
        NOTEBOOK_CONFIG.smote_k_neighbors
        if best_balance_strategy == "smote"
        else None
    ),
    scoring=NOTEBOOK_BAYES_SCORING,
    classification_threshold=SELECTED_THRESHOLD,
)

print_metric_block("=== 训练集 OOF 性能（选定阈值） ===", selected_oof_metrics)
print_metric_block("=== 测试集性能（选定阈值） ===", selected_test_metrics)
print("测试集混淆矩阵（选定阈值）:")
print(selected_test_confusion_matrix)
print(f"模型已保存到: {saved_artifacts['model_path']}")
print(f"预处理包已保存到: {saved_artifacts['preprocessor_path']}")
print(f"推理清单已保存到: {saved_artifacts['manifest_path']}")
